# Champion AHP: Simulation and Analysis

This notebook demonstrates the IPCM construction methods and reproduces the analysis from:

> Sokantika & Ratnapinda (2025). *The Role of Ranking Information in AHP with Limited Pairwise Comparisons*

**For exact replication of paper results, use the pre-generated data from Zenodo:**
https://doi.org/10.5281/zenodo.17404548

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kendalltau
from scipy.spatial.distance import euclidean

from src import (
    # PCM utilities
    generate_consistent_PCM,
    add_noise_linear_scale,
    calculate_consistency_ratio,
    llsm_complete_pcm,
    geometric_mean_incomplete_pcm_spanning_trees,
    # IPCM methods
    create_incomplete_pcm_star,
    create_incomplete_pcm_cycle,
    create_incomplete_pcm_AHP_express,
    create_incomplete_pcm_tournament,
    create_incomplete_pcm_tournament_champion_closure,
    # Visualization
    create_performance_summary,
    plot_mean_euclidean_line,
    plot_mean_kendall_line,
    ALG_NAME_MAPPING,
    # Analysis
    run_analysis,
    print_summary_table,
    print_manuscript_summary,
)

np.random.seed(42)

## 1. Basic Example: Single PCM

In [ ]:
# Generate a 5-alternative PCM with inconsistency
n = 5
sigma = 1.0

pcm, true_weights = generate_consistent_PCM(n)
noisy_pcm = add_noise_linear_scale(pcm, sigma)

print(f"True weights: {true_weights.round(3)}")
print(f"Consistency Ratio: {calculate_consistency_ratio(noisy_pcm):.4f}")

In [ ]:
# Apply each IPCM method
methods = {
    'Star': create_incomplete_pcm_star(noisy_pcm),
    'Cycle': create_incomplete_pcm_cycle(noisy_pcm),
    'AHP Express': create_incomplete_pcm_AHP_express(noisy_pcm, true_weights),
    'C-AHP': create_incomplete_pcm_tournament(noisy_pcm),
    'CC-AHP': create_incomplete_pcm_tournament_champion_closure(noisy_pcm)
}

# Benchmark from complete PCM
benchmark = llsm_complete_pcm(noisy_pcm)
print(f"Benchmark weights: {benchmark.round(3)}")
print()

# Estimate weights from each IPCM
for name, ipcm in methods.items():
    weights = geometric_mean_incomplete_pcm_spanning_trees(ipcm)
    euc = euclidean(true_weights, weights)
    tau, _ = kendalltau(true_weights, weights)
    print(f"{name:12}: Euclidean={euc:.4f}, Kendall's tau={tau:.3f}")

## 2. Monte Carlo Simulation

For paper replication, use `n_samples=10000`. Here we use a smaller sample for demonstration.

In [ ]:
def run_simulation(n_alt, sigma, n_samples=500):
    """Run simulation for one condition."""
    records = []
    
    for i in range(n_samples):
        pcm, true_w = generate_consistent_PCM(n_alt)
        noisy = add_noise_linear_scale(pcm, sigma)
        
        # Benchmark (complete PCM)
        bench_w = llsm_complete_pcm(noisy)
        euc = euclidean(true_w, bench_w)
        tau, _ = kendalltau(true_w, bench_w)
        records.append({
            'Sample': i, 'Alternatives': n_alt, 'Sigma': sigma,
            'Algorithm': 'GeometricMean',
            'EuclideanDistance': euc, 'KendallsTau': tau
        })
        
        # IPCM methods
        ipcm_methods = [
            ('create_incomplete_pcm_star', create_incomplete_pcm_star(noisy)),
            ('create_incomplete_pcm_cycle', create_incomplete_pcm_cycle(noisy)),
            ('create_incomplete_pcm_AHP_express', create_incomplete_pcm_AHP_express(noisy, true_w)),
            ('create_incomplete_pcm_tournament', create_incomplete_pcm_tournament(noisy)),
            ('create_incomplete_pcm_tournament_champion_closure', create_incomplete_pcm_tournament_champion_closure(noisy)),
        ]
        
        for alg_name, ipcm in ipcm_methods:
            w = geometric_mean_incomplete_pcm_spanning_trees(ipcm)
            if np.sum(w) > 0:
                euc = euclidean(true_w, w)
                tau, _ = kendalltau(true_w, w)
                records.append({
                    'Sample': i, 'Alternatives': n_alt, 'Sigma': sigma,
                    'Algorithm': alg_name,
                    'EuclideanDistance': euc, 'KendallsTau': tau
                })
    
    return pd.DataFrame(records)

In [ ]:
# Run simulation across conditions (use smaller n_samples for quick demo)
n_samples = 500  # Use 10000 for paper replication

all_results = []
for n_alt in [4, 5, 6, 7]:
    for sigma in [0.5, 1.5, 2.5]:
        print(f"Running n={n_alt}, σ={sigma}...")
        df = run_simulation(n_alt, sigma, n_samples)
        all_results.append(df)

df_performance = pd.concat(all_results, ignore_index=True)
print(f"\nTotal records: {len(df_performance)}")

## 3. Visualization

In [ ]:
# Create performance summary
performance_summary = create_performance_summary(df_performance)
performance_summary.head(10)

In [ ]:
# Plot Mean Euclidean Distance
plot_mean_euclidean_line(performance_summary, selected_sigmas=[0.5, 1.5, 2.5])

In [ ]:
# Plot Mean Kendall's Tau
plot_mean_kendall_line(performance_summary, selected_sigmas=[0.5, 1.5, 2.5])

## 4. Statistical Analysis

Friedman test with Kendall's W effect size and Conover post-hoc with Compact Letter Display (CLD).

In [ ]:
# Run analysis for Euclidean Distance
results_euclidean = run_analysis(df_performance, metric='EuclideanDistance')
print_summary_table(results_euclidean, metric='EuclideanDistance')

In [ ]:
# Run analysis for Kendall's Tau
results_kendall = run_analysis(df_performance, metric='KendallsTau')
print_summary_table(results_kendall, metric='KendallsTau')

In [ ]:
# Print manuscript-ready summary
print_manuscript_summary(results_euclidean, results_kendall)

## 5. Key Findings

1. **Weight Accuracy (Euclidean Distance):** CC-AHP outperforms other methods under high inconsistency (σ ≥ 1.5) or larger n (≥ 6).

2. **Ranking Accuracy (Kendall's τ):** Effect sizes (Kendall's W) are negligible across methods, indicating no reliable practical advantage.

3. **Transition Pattern:** Cycle performs best for small n and low σ; CC-AHP becomes superior as complexity increases.

4. **Local Ranking Information:** C-AHP and CC-AHP leverage tournament structure to embed local ranking cues without requiring prior global ranking knowledge.